# 01 — Pandas Foundations

**Context:** Each exercise uses a realistic business dataset. No solutions are provided — use the [pandas documentation](https://pandas.pydata.org/docs/) exclusively.

**How to check your work:** Run the `assert` block at the end of each exercise. A silent pass means correct. An `AssertionError` tells you what's wrong.

**Allowed libraries:** `pandas`, `numpy`


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml

# --- Dataset: IBM HR Analytics (Employee Attrition) ---
# 1470 employees, 35 features. Business context: HR analytics at a mid-size firm.
hr = fetch_openml(name='ibm-employee-attrition', version=1, as_frame=True, parser='auto').frame
hr.head()

---
## Exercise 1 — Indexing & Selection

**Business question:** The HR director wants a focused view of Sales department employees who earn above the company median monthly income.

1. Select all rows where `Department == 'Sales'` AND `MonthlyIncome` is above the **overall** median monthly income.
2. Return only the columns: `EmployeeNumber`, `Department`, `JobRole`, `MonthlyIncome`, `Attrition`.
3. Assign the result to `sales_high_earners`.
4. Reset the index (drop the old one).

In [ ]:
def get_sales_high_earners(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns Sales dept employees above overall median MonthlyIncome.
    Columns: EmployeeNumber, Department, JobRole, MonthlyIncome, Attrition.
    Index reset, old index dropped.
    """
    # YOUR CODE HERE
    pass

sales_high_earners = get_sales_high_earners(hr)

In [ ]:
# --- ASSERTIONS ---
assert isinstance(sales_high_earners, pd.DataFrame), "Must return a DataFrame"
assert list(sales_high_earners.columns) == ['EmployeeNumber', 'Department', 'JobRole', 'MonthlyIncome', 'Attrition'], "Wrong columns or order"
assert (sales_high_earners['Department'] == 'Sales').all(), "All rows must be Sales"
assert (sales_high_earners['MonthlyIncome'] > hr['MonthlyIncome'].median()).all(), "All rows must exceed overall median"
assert sales_high_earners.index.tolist() == list(range(len(sales_high_earners))), "Index must be reset"
print(f"✓ Exercise 1 passed — {len(sales_high_earners)} rows")

---
## Exercise 2 — Data Types & Missing Values

**Business question:** Before building any model, the data team needs a quality report on the dataset.

1. Write a function `data_quality_report(df)` that returns a DataFrame with one row per column containing:
   - `dtype`: the pandas dtype
   - `null_count`: number of nulls
   - `null_pct`: percentage of nulls (0–100, rounded to 2 decimals)
   - `n_unique`: number of unique values
2. The result index should be the column names.
3. Sort by `null_pct` descending.

In [ ]:
def data_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a quality report DataFrame indexed by column name.
    Columns: dtype, null_count, null_pct, n_unique.
    Sorted by null_pct descending.
    """
    # YOUR CODE HERE
    pass

report = data_quality_report(hr)

In [ ]:
# --- ASSERTIONS ---
assert isinstance(report, pd.DataFrame), "Must return a DataFrame"
assert list(report.columns) == ['dtype', 'null_count', 'null_pct', 'n_unique'], "Wrong columns"
assert report.index.tolist() == list(hr.columns), "Index must be column names"
assert report['null_pct'].is_monotonic_decreasing, "Must be sorted by null_pct descending"
assert report.loc['MonthlyIncome', 'n_unique'] == hr['MonthlyIncome'].nunique(), "n_unique wrong for MonthlyIncome"
print("✓ Exercise 2 passed")

---
## Exercise 3 — GroupBy & Aggregation

**Business question:** The compensation team wants a summary table to identify pay equity issues across departments and job levels.

1. Group by `Department` and `JobLevel`.
2. Compute for `MonthlyIncome`: `mean`, `median`, `std`, `min`, `max`, and `count`.
3. Round all numeric results to 2 decimal places.
4. Flatten the column MultiIndex into single strings like `MonthlyIncome_mean`, `MonthlyIncome_std`, etc.
5. Assign to `compensation_summary`. Reset index.

In [ ]:
def get_compensation_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    GroupBy Department + JobLevel, aggregate MonthlyIncome.
    Flat column names: MonthlyIncome_mean, etc.
    """
    # YOUR CODE HERE
    pass

compensation_summary = get_compensation_summary(hr)

In [ ]:
# --- ASSERTIONS ---
expected_cols = ['Department', 'JobLevel', 'MonthlyIncome_mean', 'MonthlyIncome_median',
                 'MonthlyIncome_std', 'MonthlyIncome_min', 'MonthlyIncome_max', 'MonthlyIncome_count']
assert list(compensation_summary.columns) == expected_cols, f"Columns mismatch: {list(compensation_summary.columns)}"
assert compensation_summary['MonthlyIncome_count'].dtype in [np.int64, np.float64, 'int64', 'float64'], "count must be numeric"
assert compensation_summary['MonthlyIncome_mean'].round(2).equals(compensation_summary['MonthlyIncome_mean']), "Values must be rounded to 2dp"
print(f"✓ Exercise 3 passed — {len(compensation_summary)} groups")

---
## Exercise 4 — Merging DataFrames

**Business question:** The HR system stores performance reviews separately from the main employee file. You need to combine them and flag data issues.

First, run the cell below to create the two tables.

In [ ]:
# Setup — do not modify
np.random.seed(42)
employee_subset = hr[['EmployeeNumber', 'Department', 'MonthlyIncome', 'Attrition']].copy()

# Performance reviews — only exists for 80% of employees, some duplicates
sampled_ids = hr['EmployeeNumber'].sample(frac=0.8, random_state=42)
performance = pd.DataFrame({
    'emp_id': sampled_ids.values,
    'performance_score': np.random.randint(1, 6, size=len(sampled_ids)),
    'review_year': np.random.choice([2022, 2023], size=len(sampled_ids))
})
print(f"Employees: {len(employee_subset)} | Reviews: {len(performance)}")

1. Merge `employee_subset` with `performance` so that **all employees are kept**, even those without a review. Match on `EmployeeNumber` ↔ `emp_id`.
2. Add a boolean column `has_review` that is `True` when a performance record exists.
3. For employees without a review, fill `performance_score` with `0` and `review_year` with `0` (as integers).
4. Assign to `employee_reviews`.

In [ ]:
def merge_employee_reviews(employees: pd.DataFrame, reviews: pd.DataFrame) -> pd.DataFrame:
    """
    Left merge employees with reviews.
    Add has_review bool. Fill missing scores/years with 0.
    """
    # YOUR CODE HERE
    pass

employee_reviews = merge_employee_reviews(employee_subset, performance)

In [ ]:
# --- ASSERTIONS ---
assert len(employee_reviews) >= len(employee_subset), "All employees must be retained"
assert 'has_review' in employee_reviews.columns, "Missing has_review column"
assert employee_reviews['has_review'].dtype == bool, "has_review must be bool"
assert employee_reviews['performance_score'].isna().sum() == 0, "No nulls allowed in performance_score"
assert employee_reviews['review_year'].isna().sum() == 0, "No nulls allowed in review_year"
assert employee_reviews.loc[~employee_reviews['has_review'], 'performance_score'].eq(0).all(), "Missing reviews must have score=0"
print(f"✓ Exercise 4 passed — {employee_reviews['has_review'].sum()} employees with reviews")

---
## Exercise 5 — Apply, Map & Vectorized Operations

**Business question:** The comp team needs derived salary bands and tenure classifications.

Working on the original `hr` DataFrame, create a new DataFrame `hr_enriched` (copy of `hr`) with these added columns:

1. `salary_band`: categorize `MonthlyIncome` into `'Low'` (< 3000), `'Mid'` (3000–7999), `'High'` (8000–14999), `'Executive'` (≥ 15000).
2. `tenure_group`: categorize `YearsAtCompany` into `'New'` (0–2), `'Developing'` (3–5), `'Established'` (6–10), `'Veteran'` (> 10).
3. `income_vs_avg`: each employee's `MonthlyIncome` minus the **mean income of their Department** (i.e., department-relative income).
4. Do **not** use any Python `for` loops — use pandas vectorized operations or `pd.cut`.

In [ ]:
def enrich_hr_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns copy of df with salary_band, tenure_group, income_vs_avg columns.
    No for loops.
    """
    # YOUR CODE HERE
    pass

hr_enriched = enrich_hr_data(hr)

In [ ]:
# --- ASSERTIONS ---
for col in ['salary_band', 'tenure_group', 'income_vs_avg']:
    assert col in hr_enriched.columns, f"Missing column: {col}"

assert set(hr_enriched['salary_band'].unique()).issubset({'Low', 'Mid', 'High', 'Executive'}), "Invalid salary_band values"
assert set(hr_enriched['tenure_group'].unique()).issubset({'New', 'Developing', 'Established', 'Veteran'}), "Invalid tenure_group values"

# income_vs_avg check: within each dept, values should sum to ~0
dept_check = hr_enriched.groupby('Department')['income_vs_avg'].mean().abs()
assert (dept_check < 0.01).all(), "income_vs_avg must be relative to department mean"

assert len(hr_enriched) == len(hr), "Must not change row count"
print("✓ Exercise 5 passed")
print(hr_enriched[['MonthlyIncome', 'salary_band', 'YearsAtCompany', 'tenure_group', 'income_vs_avg']].head())

---
## Exercise 6 — Reshaping: Pivot Tables

**Business question:** The HR VP wants a cross-tab showing average `MonthlyIncome` by `Department` (rows) and `JobLevel` (columns), with a grand total column.

1. Build a pivot table: rows = `Department`, columns = `JobLevel`, values = mean `MonthlyIncome`, rounded to 0 decimals (integers).
2. Add a column `Overall_Mean` = mean across all job levels for that department (row mean).
3. Sort rows by `Overall_Mean` descending.
4. Assign to `income_pivot`. Fill any NaN cells with `0`.

In [ ]:
def build_income_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pivot: Department x JobLevel, values = mean MonthlyIncome.
    Adds Overall_Mean column. Sorted descending by Overall_Mean.
    """
    # YOUR CODE HERE
    pass

income_pivot = build_income_pivot(hr)

In [ ]:
# --- ASSERTIONS ---
assert isinstance(income_pivot, pd.DataFrame)
assert 'Overall_Mean' in income_pivot.columns, "Missing Overall_Mean column"
assert income_pivot.index.name == 'Department', "Row index must be Department"
assert income_pivot['Overall_Mean'].is_monotonic_decreasing, "Must be sorted by Overall_Mean descending"
assert income_pivot.isna().sum().sum() == 0, "No NaN values allowed"
print("✓ Exercise 6 passed")
print(income_pivot)

---
## Exercise 7 — GroupBy Transform & Custom Aggregation

**Business question:** Identify employees who are outliers within their own job role — specifically those whose monthly income deviates more than 1.5 standard deviations from their job role's mean.

1. Add a column `income_zscore_by_role` to a copy of `hr`: the z-score of `MonthlyIncome` **within each `JobRole`** (use transform).
2. Add a boolean column `is_pay_outlier`: `True` if `abs(income_zscore_by_role) > 1.5`.
3. Return a summary DataFrame `outlier_summary` grouped by `JobRole` with columns: `total_employees`, `n_outliers`, `outlier_pct` (rounded to 2 decimals).
4. Sort by `outlier_pct` descending.

In [ ]:
def get_pay_outlier_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns outlier_summary: JobRole x [total_employees, n_outliers, outlier_pct].
    Sorted by outlier_pct descending.
    """
    # YOUR CODE HERE
    pass

outlier_summary = get_pay_outlier_summary(hr)

In [ ]:
# --- ASSERTIONS ---
assert list(outlier_summary.columns) == ['total_employees', 'n_outliers', 'outlier_pct'], "Wrong columns"
assert outlier_summary.index.name == 'JobRole', "Index must be JobRole"
assert outlier_summary['outlier_pct'].is_monotonic_decreasing, "Must sort by outlier_pct descending"
assert outlier_summary['total_employees'].sum() == len(hr), "total_employees must sum to full dataset size"
assert (outlier_summary['outlier_pct'] <= 100).all(), "outlier_pct can't exceed 100"
print("✓ Exercise 7 passed")
print(outlier_summary.head())

---
## Exercise 8 — String Operations & Categorical Data

**Business question:** Prepare a clean categorical encoding report for the ML team, summarizing all object/categorical columns.

1. Identify all columns in `hr` with dtype `object`.
2. For each such column, create a frequency table as a DataFrame with columns: `value`, `count`, `pct` (% of total rows, rounded to 2dp).
3. Return a **dict** `freq_tables` where keys are column names and values are these frequency DataFrames, sorted by `count` descending.
4. Additionally, return `high_cardinality_cols`: a list of column names where `n_unique > 5`, sorted alphabetically.

In [ ]:
def get_categorical_summary(df: pd.DataFrame):
    """
    Returns:
      freq_tables: dict of {col_name: DataFrame(value, count, pct)}
      high_cardinality_cols: list of object cols with n_unique > 5
    """
    # YOUR CODE HERE
    pass

freq_tables, high_cardinality_cols = get_categorical_summary(hr)

In [ ]:
# --- ASSERTIONS ---
obj_cols = hr.select_dtypes(include='object').columns.tolist()
assert set(freq_tables.keys()) == set(obj_cols), "freq_tables keys must match object columns"

for col, tbl in freq_tables.items():
    assert list(tbl.columns) == ['value', 'count', 'pct'], f"Wrong columns for {col}"
    assert tbl['count'].is_monotonic_decreasing, f"{col} must be sorted by count descending"
    assert abs(tbl['pct'].sum() - 100.0) < 0.1, f"{col} pct must sum to ~100"

assert isinstance(high_cardinality_cols, list), "high_cardinality_cols must be a list"
assert high_cardinality_cols == sorted(high_cardinality_cols), "Must be sorted alphabetically"
assert all(hr[c].nunique() > 5 for c in high_cardinality_cols), "All listed cols must have n_unique > 5"
print(f"✓ Exercise 8 passed — {len(high_cardinality_cols)} high-cardinality columns: {high_cardinality_cols}")